In [10]:
"""
Myerson-Restricted Gately Feature Attributions
Tested on: lung_cancer.csv

- Players: features (columns except target)
- Game v(S):
    For classification:
      v(S) = mean predicted probability of the positive class on X_test
             when only features in S are kept (other features set to baseline mean).

- Communication Graph G:
    Undirected graph where features are connected if
    |corr(i, j)| >= CORR_THRESHOLD on the training set (standardized).

- Myerson-Restricted Game v^G:
    For any S, v^G(S) = sum of v(C) over connected components C of G[S].

- Myerson-Restricted Gately Allocation:
    x_i = s_i^G + λ (M_i^G - s_i^G),
    where
        s_i^G = v^G({i})
        M_i^G = v^G(N) - v^G(N \ {i})
        λ = (v^G(N) - Σ s_i^G) / (Σ M_i^G - Σ s_i^G)

Requirements:
    pip install numpy pandas scikit-learn networkx
"""

import numpy as np
import pandas as pd
import warnings
import networkx as nx

from typing import Dict, Iterable, FrozenSet, Tuple
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_absolute_error
from sklearn.utils.multiclass import type_of_target

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

COMMON_TARGET_NAMES = [
    "target", "label", "y", "class", "outcome",
    "cvd", "cardio", "TenYearCHD", "heart_disease", "stroke", "risk",
    "cancer", "lung_cancer", "has_cancer", "diagnosis"
]

CORR_THRESHOLD = 0.3  # threshold for correlation-based edges in G


def drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df[[c for c in df.columns if not c.lower().startswith("unnamed")]].copy()


def detect_target_column(df: pd.DataFrame, user_hint: str = None) -> str:
    if user_hint is not None and user_hint in df.columns:
        return user_hint
    lower_map = {c.lower(): c for c in df.columns}
    for name in COMMON_TARGET_NAMES:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    # fallback: last column
    return df.columns[-1]


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number]).copy()
    for c in df.columns:
        if c not in num.columns:
            try:
                num[c] = pd.to_numeric(df[c], errors="raise")
            except Exception:
                # non-numeric column, drop it silently
                pass
    return num


class FeatureGame:
    """
    Cooperative game for feature attribution:
        - Players = feature names
        - v(S) = dataset-level utility when only features in S are used
                 (others set to baseline mean).
    """

    def __init__(self,
                 model,
                 X_train: pd.DataFrame,
                 X_test: pd.DataFrame,
                 y_train: np.ndarray,
                 y_test: np.ndarray,
                 task: str,
                 feature_names: Iterable[str]):
        self.model = model
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.task = task
        self.features = list(feature_names)

        # Baseline (in scaled space): mean over training set
        self.baseline = X_train.mean(axis=0)

        # Precompute base value v({all features}) if needed
        self._cache: Dict[FrozenSet[str], float] = {}

        # For classification, determine what "utility" is
        if self.task == "classification":
            # We will treat the "positive class" as class 1 if present,
            # otherwise we use mean over "predicted class" prob.
            self.classes_ = getattr(self.model, "classes_", None)
            if self.classes_ is not None and len(self.classes_) == 2:
                # Two-class case: positive = class with label 1 if available, else index 1
                if 1 in self.classes_:
                    self.pos_index = list(self.classes_).index(1)
                else:
                    self.pos_index = 1
            else:
                self.pos_index = None  # multiclass
        else:
            self.classes_ = None
            self.pos_index = None

    def _coalition_matrix(self, S: Iterable[str], use_test: bool = True) -> pd.DataFrame:
        """
        Build masked dataset where:
          - features in S use their actual values
          - features not in S are set to baseline mean (from X_train).
        """
        if use_test:
            X = self.X_test.copy()
        else:
            X = self.X_train.copy()

        S = set(S)
        for feat in self.features:
            if feat not in S:
                X[feat] = self.baseline[feat]
        return X

    def _utility_from_predictions(self, X_masked: pd.DataFrame) -> float:
        """
        Map predictions to a scalar coalition value v(S).
        """
        if self.task == "classification":
            proba = self.model.predict_proba(X_masked.values)
            if self.pos_index is not None and self.pos_index < proba.shape[1]:
                # mean probability of positive class
                return float(np.mean(proba[:, self.pos_index]))
            else:
                # multiclass: mean max probability (confidence)
                return float(np.mean(np.max(proba, axis=1)))
        else:  # regression
            preds = self.model.predict(X_masked.values)
            # Here we choose mean prediction as a simple scalar utility
            return float(np.mean(preds))

    def v(self, S: Iterable[str]) -> float:
        """
        Original characteristic function v(S).
        """
        key = frozenset(S)
        if key in self._cache:
            return self._cache[key]
        if len(key) == 0:
            # Define v(empty) as baseline prediction with all features at baseline
            X_masked = self._coalition_matrix([], use_test=True)
            val = self._utility_from_predictions(X_masked)
            self._cache[key] = val
            return val

        X_masked = self._coalition_matrix(key, use_test=True)
        val = self._utility_from_predictions(X_masked)
        self._cache[key] = val
        return val


def myerson_restricted_value(S: Iterable[str], game: FeatureGame, G: nx.Graph) -> float:
    """
    Myerson restricted game:
        v^G(S) = sum over connected components C of G[S] of v(C).
    """
    key = frozenset(S)
    if len(key) == 0:
        return 0.0

    subG = G.subgraph(key)
    total = 0.0
    for comp in nx.connected_components(subG):
        total += game.v(comp)
    return total


def gately_allocation_myerson(features: Iterable[str],
                              game: FeatureGame,
                              G: nx.Graph) -> Dict[str, float]:
    """
    Myerson-Restricted Gately allocation:
      - Players = features
      - Characteristic function = v^G derived from game.v and graph G
    """
    features = list(features)
    N = frozenset(features)

    def vG(S: Iterable[str]) -> float:
        return myerson_restricted_value(S, game, G)

    # Worth of grand coalition
    vN = vG(N)

    # Singleton and "leave-one-out" values
    s: Dict[str, float] = {}
    M: Dict[str, float] = {}

    for i in features:
        s[i] = vG({i})

    for i in features:
        N_minus_i = frozenset(j for j in features if j != i)
        M[i] = vN - vG(N_minus_i)

    sum_s = sum(s.values())
    sum_M = sum(M.values())
    denom = sum_M - sum_s

    if abs(denom) < 1e-12:
        raise ValueError("Myerson-Gately undefined: denominator near zero (degenerate game).")

    lam = (vN - sum_s) / denom

    x: Dict[str, float] = {}
    for i in features:
        x[i] = s[i] + lam * (M[i] - s[i])

    return x, s, M, vN, lam


def run_myerson_gately_on_csv(csv_path: str,
                              target_hint: str = None,
                              test_size: float = 0.25):
    print("\n" + "=" * 72)
    print(f"Myerson-Restricted Gately Feature Attributions on: {csv_path}")
    print("=" * 72)

    try:
        raw = pd.read_csv(csv_path)
    except Exception as e:
        print(f"ERROR reading CSV: {e}")
        return

    raw = drop_unnamed(raw)
    tgt_col = detect_target_column(raw, target_hint)
    if tgt_col not in raw.columns:
        print(f"ERROR: target column '{tgt_col}' not found.")
        return

    y_raw = raw[tgt_col].copy()
    X_raw = raw.drop(columns=[tgt_col])

    X_num = to_numeric_df(X_raw)
    dropped = set(X_raw.columns) - set(X_num.columns)
    if dropped:
        print(f"Note: Dropping non-numeric columns: {sorted(list(dropped))}")

    # Handle inf / NaN
    X_num = X_num.replace([np.inf, -np.inf], np.nan)
    X_num = X_num.fillna(X_num.median(numeric_only=True))

    # Detect task type
    y_type = type_of_target(y_raw)
    if y_type in ["binary", "multiclass"]:
        task = "classification"
        # Encode labels if not numeric
        if not np.issubdtype(y_raw.dtype, np.number):
            le = LabelEncoder()
            y_vec = le.fit_transform(y_raw.values)
        else:
            y_vec = y_raw.astype(int).values
    else:
        task = "regression"
        y_vec = pd.to_numeric(y_raw, errors="coerce").values
        if np.isnan(y_vec).any():
            print("ERROR: target has non-numeric values that cannot be coerced.")
            return

    strat = y_vec if (task == "classification" and len(np.unique(y_vec)) > 1) else None

    X_train, X_test, y_train, y_test = train_test_split(
        X_num, y_vec, test_size=test_size, random_state=RANDOM_SEED, stratify=strat
    )

    # Standardize features
    scaler = StandardScaler()
    Xtr = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    Xte = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )

    feature_names = list(Xtr.columns)

    if task == "classification":
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=10,
            random_state=RANDOM_SEED
        )
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        acc = accuracy_score(y_test, y_pred)
        f1m = f1_score(y_test, y_pred, average="macro")
        print(f"Trained RandomForest (classification): acc={acc:.4f}, macroF1={f1m:.4f}")
    else:
        model = RandomForestRegressor(
            n_estimators=400,
            max_depth=10,
            random_state=RANDOM_SEED
        )
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        print(f"Trained RandomForest (regression): R2={r2:.4f}, MAE={mae:.4f}")

    print("\n" + "-" * 64)
    print("Building communication graph G (correlation-based)")
    print("-" * 64)

    corr = Xtr.corr()
    G = nx.Graph()
    G.add_nodes_from(feature_names)

    for i, fi in enumerate(feature_names):
        for j in range(i + 1, len(feature_names)):
            fj = feature_names[j]
            if abs(corr.loc[fi, fj]) >= CORR_THRESHOLD:
                G.add_edge(fi, fj)

    print(f"Graph G: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    comps = list(nx.connected_components(G))
    print(f"Connected components (feature clusters):")
    for idx, comp in enumerate(comps, 1):
        print(f"  Component {idx}: {sorted(list(comp))}")

    print("\n" + "-" * 64)
    print("Defining feature game v(S) and Myerson-restricted Gately allocation")
    print("-" * 64)

    game = FeatureGame(
        model=model,
        X_train=Xtr,
        X_test=Xte,
        y_train=y_train,
        y_test=y_test,
        task=task,
        feature_names=feature_names
    )

    x, s, M, vN, lam = gately_allocation_myerson(feature_names, game, G)

    print(f"\nGrand coalition value v^G(N): {vN:.6f}")
    print(f"λ (lambda) used in Gately formula: {lam:.6f}\n")

    rows = []
    for feat in feature_names:
        rows.append({
            "feature": feat,
            "vG({i})": s[feat],
            "M_i^G": M[feat],
            "x_i (Myerson-Gately)": x[feat],
        })

    df_res = pd.DataFrame(rows).set_index("feature")
    df_res["x_i (normalized)"] = df_res["x_i (Myerson-Gately)"] / df_res["x_i (Myerson-Gately)"].abs().sum()

    print("Myerson-Restricted Gately Attributions (global, dataset-level):")
    print(df_res.sort_values("x_i (Myerson-Gately)", ascending=False).round(6).to_string())


def main():
    run_myerson_gately_on_csv("../../datasets/lung_cancer_risk.csv", target_hint="lung_cancer_risk")


if __name__ == "__main__":
    main()


Myerson-Restricted Gately Feature Attributions on: ../datasets/lung_cancer_risk.csv
✓ Trained RandomForest (regression): R2=0.7195, MAE=1.4996

----------------------------------------------------------------
Building communication graph G (correlation-based)
----------------------------------------------------------------
Graph G: 3 nodes, 2 edges
Connected components (feature clusters):
  Component 1: ['drink_coffee', 'smoking', 'stress']

----------------------------------------------------------------
Defining feature game v(S) and Myerson-restricted Gately allocation
----------------------------------------------------------------

Grand coalition value v^G(N): 15.987747
λ (lambda) used in Gately formula: 0.500093

Myerson-Restricted Gately Attributions (global, dataset-level):
                vG({i})      M_i^G  x_i (Myerson-Gately)  x_i (normalized)
feature                                                                   
stress        16.193412   0.057679              8.12404

In [12]:
"""
Myerson-Restricted Gately Feature Attributions
Tested on: lung_cancer.csv

- Players: features (columns except target)
- Game v(S):
    For classification:
      v(S) = mean predicted probability of the positive class on X_test
             when only features in S are kept (other features set to baseline mean).

- Communication Graph G:
    Undirected graph where features are connected if
    |corr(i, j)| >= CORR_THRESHOLD on the training set (standardized).

- Myerson-Restricted Game v^G:
    For any S, v^G(S) = sum of v(C) over connected components C of G[S].

- Myerson-Restricted Gately Allocation:
    x_i = s_i^G + λ (M_i^G - s_i^G),
    where
        s_i^G = v^G({i})
        M_i^G = v^G(N) - v^G(N \ {i})
        λ = (v^G(N) - Σ s_i^G) / (Σ M_i^G - Σ s_i^G)

Requirements:
    pip install numpy pandas scikit-learn networkx
"""

import numpy as np
import pandas as pd
import warnings
import networkx as nx

from typing import Dict, Iterable, FrozenSet, Tuple
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, f1_score, r2_score, mean_absolute_error
from sklearn.utils.multiclass import type_of_target

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

COMMON_TARGET_NAMES = [
    "target", "label", "y", "class", "outcome",
    "cvd", "cardio", "TenYearCHD", "heart_disease", "stroke", "risk",
    "cancer", "lung_cancer", "has_cancer", "diagnosis"
]

CORR_THRESHOLD = 0.5  # threshold for correlation-based edges in G


def drop_unnamed(df: pd.DataFrame) -> pd.DataFrame:
    return df[[c for c in df.columns if not c.lower().startswith("unnamed")]].copy()


def detect_target_column(df: pd.DataFrame, user_hint: str = None) -> str:
    if user_hint is not None and user_hint in df.columns:
        return user_hint
    lower_map = {c.lower(): c for c in df.columns}
    for name in COMMON_TARGET_NAMES:
        if name.lower() in lower_map:
            return lower_map[name.lower()]
    # fallback: last column
    return df.columns[-1]


def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    num = df.select_dtypes(include=[np.number]).copy()
    for c in df.columns:
        if c not in num.columns:
            try:
                num[c] = pd.to_numeric(df[c], errors="raise")
            except Exception:
                # non-numeric column, drop it silently
                pass
    return num


class FeatureGame:
    """
    Cooperative game for feature attribution:
        - Players = feature names
        - v(S) = dataset-level utility when only features in S are used
                 (others set to baseline mean).
    """

    def __init__(self,
                 model,
                 X_train: pd.DataFrame,
                 X_test: pd.DataFrame,
                 y_train: np.ndarray,
                 y_test: np.ndarray,
                 task: str,
                 feature_names: Iterable[str]):
        self.model = model
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.task = task
        self.features = list(feature_names)

        # Baseline (in scaled space): mean over training set
        self.baseline = X_train.mean(axis=0)

        # Precompute base value v({all features}) if needed
        self._cache: Dict[FrozenSet[str], float] = {}

        # For classification, determine what "utility" is
        if self.task == "classification":
            # We will treat the "positive class" as class 1 if present,
            # otherwise we use mean over "predicted class" prob.
            self.classes_ = getattr(self.model, "classes_", None)
            if self.classes_ is not None and len(self.classes_) == 2:
                # Two-class case: positive = class with label 1 if available, else index 1
                if 1 in self.classes_:
                    self.pos_index = list(self.classes_).index(1)
                else:
                    self.pos_index = 1
            else:
                self.pos_index = None  # multiclass
        else:
            self.classes_ = None
            self.pos_index = None

    def _coalition_matrix(self, S: Iterable[str], use_test: bool = True) -> pd.DataFrame:
        """
        Build masked dataset where:
          - features in S use their actual values
          - features not in S are set to baseline mean (from X_train).
        """
        if use_test:
            X = self.X_test.copy()
        else:
            X = self.X_train.copy()

        S = set(S)
        for feat in self.features:
            if feat not in S:
                X[feat] = self.baseline[feat]
        return X

    def _utility_from_predictions(self, X_masked: pd.DataFrame) -> float:
        """
        Map predictions to a scalar coalition value v(S).
        """
        if self.task == "classification":
            proba = self.model.predict_proba(X_masked.values)
            if self.pos_index is not None and self.pos_index < proba.shape[1]:
                # mean probability of positive class
                return float(np.mean(proba[:, self.pos_index]))
            else:
                # multiclass: mean max probability (confidence)
                return float(np.mean(np.max(proba, axis=1)))
        else:  # regression
            preds = self.model.predict(X_masked.values)
            # Here we choose mean prediction as a simple scalar utility
            return float(np.mean(preds))

    def v(self, S: Iterable[str]) -> float:
        """
        Original characteristic function v(S).
        """
        key = frozenset(S)
        if key in self._cache:
            return self._cache[key]
        if len(key) == 0:
            # Define v(empty) as baseline prediction with all features at baseline
            X_masked = self._coalition_matrix([], use_test=True)
            val = self._utility_from_predictions(X_masked)
            self._cache[key] = val
            return val

        X_masked = self._coalition_matrix(key, use_test=True)
        val = self._utility_from_predictions(X_masked)
        self._cache[key] = val
        return val


def myerson_restricted_value(S: Iterable[str], game: FeatureGame, G: nx.Graph) -> float:
    """
    Myerson restricted game:
        v^G(S) = sum over connected components C of G[S] of v(C).
    """
    key = frozenset(S)
    if len(key) == 0:
        return 0.0

    subG = G.subgraph(key)
    total = 0.0
    for comp in nx.connected_components(subG):
        total += game.v(comp)
    return total


def gately_allocation_myerson(features: Iterable[str],
                              game: FeatureGame,
                              G: nx.Graph) -> Dict[str, float]:
    """
    Myerson-Restricted Gately allocation:
      - Players = features
      - Characteristic function = v^G derived from game.v and graph G
    """
    features = list(features)
    N = frozenset(features)

    def vG(S: Iterable[str]) -> float:
        return myerson_restricted_value(S, game, G)

    # Worth of grand coalition
    vN = vG(N)

    # Singleton and "leave-one-out" values
    s: Dict[str, float] = {}
    M: Dict[str, float] = {}

    for i in features:
        s[i] = vG({i})

    for i in features:
        N_minus_i = frozenset(j for j in features if j != i)
        M[i] = vN - vG(N_minus_i)

    sum_s = sum(s.values())
    sum_M = sum(M.values())
    denom = sum_M - sum_s

    if abs(denom) < 1e-12:
        raise ValueError("Myerson-Gately undefined: denominator near zero (degenerate game).")

    lam = (vN - sum_s) / denom

    x: Dict[str, float] = {}
    for i in features:
        x[i] = s[i] + lam * (M[i] - s[i])

    return x, s, M, vN, lam


def run_myerson_gately_on_csv(csv_path: str,
                              target_hint: str = None,
                              test_size: float = 0.25):
    print("\n" + "=" * 72)
    print(f"Myerson-Restricted Gately Feature Attributions on: {csv_path}")
    print("=" * 72)

    try:
        raw = pd.read_csv(csv_path)
    except Exception as e:
        print(f"ERROR reading CSV: {e}")
        return

    raw = drop_unnamed(raw)
    tgt_col = detect_target_column(raw, target_hint)
    if tgt_col not in raw.columns:
        print(f"ERROR: target column '{tgt_col}' not found.")
        return

    y_raw = raw[tgt_col].copy()
    X_raw = raw.drop(columns=[tgt_col])

    X_num = to_numeric_df(X_raw)
    dropped = set(X_raw.columns) - set(X_num.columns)
    if dropped:
        print(f"Note: Dropping non-numeric columns: {sorted(list(dropped))}")

    # Handle inf / NaN
    X_num = X_num.replace([np.inf, -np.inf], np.nan)
    X_num = X_num.fillna(X_num.median(numeric_only=True))

    # Detect task type
    y_type = type_of_target(y_raw)
    if y_type in ["binary", "multiclass"]:
        task = "classification"
        # Encode labels if not numeric
        if not np.issubdtype(y_raw.dtype, np.number):
            le = LabelEncoder()
            y_vec = le.fit_transform(y_raw.values)
        else:
            y_vec = y_raw.astype(int).values
    else:
        task = "regression"
        y_vec = pd.to_numeric(y_raw, errors="coerce").values
        if np.isnan(y_vec).any():
            print("ERROR: target has non-numeric values that cannot be coerced.")
            return

    strat = y_vec if (task == "classification" and len(np.unique(y_vec)) > 1) else None

    X_train, X_test, y_train, y_test = train_test_split(
        X_num, y_vec, test_size=test_size, random_state=RANDOM_SEED, stratify=strat
    )

    # Standardize features
    scaler = StandardScaler()
    Xtr = pd.DataFrame(
        scaler.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index,
    )
    Xte = pd.DataFrame(
        scaler.transform(X_test),
        columns=X_test.columns,
        index=X_test.index,
    )

    feature_names = list(Xtr.columns)

    if task == "classification":
        model = RandomForestClassifier(
            n_estimators=500,
            max_depth=10,
            random_state=RANDOM_SEED
        )
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        acc = accuracy_score(y_test, y_pred)
        f1m = f1_score(y_test, y_pred, average="macro")
        print(f"Trained RandomForest (classification): acc={acc:.4f}, macroF1={f1m:.4f}")
    else:
        model = RandomForestRegressor(
            n_estimators=400,
            max_depth=10,
            random_state=RANDOM_SEED
        )
        model.fit(Xtr.values, y_train)
        y_pred = model.predict(Xte.values)
        r2 = r2_score(y_test, y_pred)
        mae = mean_absolute_error(y_test, y_pred)
        print(f"Trained RandomForest (regression): R2={r2:.4f}, MAE={mae:.4f}")

    print("\n" + "-" * 64)
    print("Building communication graph G (correlation-based)")
    print("-" * 64)

    corr = Xtr.corr()
    G = nx.Graph()
    G.add_nodes_from(feature_names)

    for i, fi in enumerate(feature_names):
        for j in range(i + 1, len(feature_names)):
            fj = feature_names[j]
            if abs(corr.loc[fi, fj]) >= CORR_THRESHOLD:
                G.add_edge(fi, fj)

    print(f"Graph G: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    comps = list(nx.connected_components(G))
    print(f"Connected components (feature clusters):")
    for idx, comp in enumerate(comps, 1):
        print(f"  Component {idx}: {sorted(list(comp))}")

    print("\n" + "-" * 64)
    print("Defining feature game v(S) and Myerson-restricted Gately allocation")
    print("-" * 64)

    game = FeatureGame(
        model=model,
        X_train=Xtr,
        X_test=Xte,
        y_train=y_train,
        y_test=y_test,
        task=task,
        feature_names=feature_names
    )

    x, s, M, vN, lam = gately_allocation_myerson(feature_names, game, G)

    print(f"\nGrand coalition value v^G(N): {vN:.6f}")
    print(f"λ (lambda) used in Gately formula: {lam:.6f}\n")

    rows = []
    for feat in feature_names:
        rows.append({
            "feature": feat,
            "vG({i})": s[feat],
            "M_i^G": M[feat],
            "x_i (Myerson-Gately)": x[feat],
        })

    df_res = pd.DataFrame(rows).set_index("feature")
    df_res["x_i (normalized)"] = df_res["x_i (Myerson-Gately)"] / df_res["x_i (Myerson-Gately)"].abs().sum()

    print("Myerson-Restricted Gately Attributions (global, dataset-level):")
    print(df_res.sort_values("x_i (Myerson-Gately)", ascending=False).round(6).to_string())


def main():
    run_myerson_gately_on_csv("../../datasets/cv_risk.csv", target_hint="cv_risk")


if __name__ == "__main__":
    main()


Myerson-Restricted Gately Feature Attributions on: ../datasets/cv_risk.csv
✓ Trained RandomForest (regression): R2=0.6503, MAE=1.3886

----------------------------------------------------------------
Building communication graph G (correlation-based)
----------------------------------------------------------------
Graph G: 4 nodes, 2 edges
Connected components (feature clusters):
  Component 1: ['bmi', 'diet_score', 'sleep_duration']
  Component 2: ['family_history']

----------------------------------------------------------------
Defining feature game v(S) and Myerson-restricted Gately allocation
----------------------------------------------------------------

Grand coalition value v^G(N): 22.781119
λ (lambda) used in Gately formula: 0.499964

Myerson-Restricted Gately Attributions (global, dataset-level):
                  vG({i})      M_i^G  x_i (Myerson-Gately)  x_i (normalized)
feature                                                                     
family_history  11.60165